In [ ]:
import numpy as np
from collections import Counter
import pandas as pd
import sys

In [12]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None, counts=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.counts = counts
    
    def is_leaf_node(self):
        return self.value is not None

In [28]:
class DecisionTree:

    def __init__(self, min_sample_split = 2, max_depth = None, n_features = None, min_info_gain = None):
        self.min_sample_split = min_sample_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.min_info_gain = min_info_gain
        self.root = None

        
    # Calculate entropy: H(S) = - \sum p(i)log_2p(i)
    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        entropy = -np.sum([p * np.log2(p) for p in ps if p > 0])

        return entropy
    

    # Calculate information gain
    def _information_gain(self, y, X, threshold):
        parent_entropy = self._entropy(y)

        # split
        left_index = np.argwhere(X <= threshold).flatten()
        right_index = np.argwhere(X > threshold).flatten()

        # if pure node, return 0 (no information gain)
        if len(left_index) == 0 or len(right_index) == 0:
            return 0

        # calculate child entropy
        n = len(y)
        n_l, n_r = len(left_index), len(right_index)
        e_l, e_r = self._entropy(y[left_index]), self._entropy(y[right_index])

        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

        return parent_entropy - child_entropy


    # find the best split rule for each node, return feature and its threshold
    def _best_split(self, X, y, feature_index):
        best_gain = -1
        split_index, split_threshold = None, None

        for feature in feature_index:
            X_column = X[:, feature]
            thresholds = np.unique(X_column)

            for threshold in thresholds:
                gain = self._information_gain(y, X_column, threshold)

                if gain > best_gain:
                    best_gain = gain
                    split_index = feature
                    split_threshold = threshold
        
        return split_index, split_threshold
    

    
    def _most_common_label(self, y):
        counter = Counter(y)

        most_common = counter.most_common(1)[0][0]

        return most_common


    def _grow_tree(self, X, y, depth = 0):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        counts_arr = np.bincount(y.astype(int), minlength=2)
        node_counts = {0: counts_arr[0], 1: counts_arr[1]}

        # Stopping criteria
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_sample_split):
            leaf_value = self._most_common_label(y)
            return Node(value = leaf_value, counts=node_counts)

        # find the best split rule
        feature_index = range(n_features)
        best_feature, best_threshold = self._best_split(X, y, feature_index) # use all the features
        if best_feature is None or (self.min_info_gain is None):
            return Node(value=self._most_common_label(y), counts=node_counts)
        
        # split the node
        left_index = np.argwhere(X[:, best_feature] <= best_threshold).flatten()
        right_index = np.argwhere(X[:, best_feature] > best_threshold).flatten()

        # recursively split
        left = self._grow_tree(X[left_index, :], y[left_index], depth + 1)
        right = self._grow_tree(X[right_index, :], y[right_index], depth + 1)

        return Node(best_feature, best_threshold, left, right, counts=node_counts)
    

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        
        return self._traverse_tree(x, node.right)

    def calculate_error(self, X_test, y_test):
        predictions = self.predict(X_test)
        accuracy = np.sum(y_test == predictions) / len(y_test)

        return 1 - accuracy
    

    def fit(self, X, y):
        y = np.array(y).flatten().astype(int)
        self.n_features = X.shape[1] if not self.n_features else min(self.n_features, X.shape[1])
        self.root = self._grow_tree(X, y)


In [26]:
def print_tree(node, features):
    def _format_counts(counts):
        if counts is None: return ""
        c0 = counts.get(0,0)
        c1 = counts.get(1,0)

        return f"[{c0} 0/{c1} 1]"
    
    def _recursive_print(curr_node, depth, split_feature_name = None, split_val =None):
        if curr_node is None:
            return 
        
        indent = "| " * depth
        info_str = ""

        if split_feature_name is not None:
            info_str += f"{indent}{split_feature_name} = {split_val}: "
        else: # root node
            info_str += indent

        info_str += _format_counts(curr_node.counts)

        print(info_str)

        if not curr_node.is_leaf_node():
            feature = features[curr_node.feature]

            _recursive_print(curr_node.left, depth + 1, feature, 0)
            _recursive_print(curr_node.right, depth + 1, feature, 1)
    
    _recursive_print(node, 0)


In [31]:
if __name__ == "__main__":

    if len(sys.argv) < 4:
        print("Usage: python decision_tree.py <train_file> <val_file> <max_depth>")
        sys.exit(1)
    
    train_file_path = 'data/' + sys.argv[1]
    val_file_path = 'data/' + sys.argv[2]
    max_depth = int(sys.argv[3])


    df_train = pd.read_csv(train_file_path, sep='\t')
    df_val = pd.read_csv(val_file_path, sep='\t')

    # print(df_train.head())

    train_data = df_train.to_numpy()
    val_data = df_val.to_numpy()

    X_train = train_data[:, :-1]
    y_train = train_data[:, -1]

    X_val = val_data[:, :-1]
    y_val = val_data[:, -1]

    tree = DecisionTree(max_depth, min_info_gain=0)

    tree.fit(X_train, y_train)

    feature_names = list(df_train.columns[:-1])
    print_tree(tree.root, feature_names)
    

Usage: python decision_tree.py <train_file> <val_file> <max_depth>


SystemExit: 1

d:\CodingProjects\10701-ML\.venv\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
